# ⚡ Sočasni delovni tokovi agentov z Microsoft Foundry (Python)

## 📋 Napredni vodič za vzporedno procesiranje

Ta zvezek prikazuje **vzorce sočasnih delovnih tokov** z uporabo Microsoft Agent Framework. Naučili se boste, kako ustvariti visoko zmogljive delovne tokove vzporednega procesiranja, kjer več AI agentov deluje istočasno, kar močno izboljša prepustnost in omogoči sofisticirane večniten poslovne procese.

> **Opomba o migraciji:** Ta primer je prej uporabljal GitHub Models. GitHub Models je zastarel (upokojuje se julija 2026), zato zdaj uporablja **Microsoft Foundry** preko `FoundryChatClient`, ki cilja na Azure OpenAI **Responses API**.

## 🎯 Cilji učenja

### 🚀 **Osnove sočasnega procesiranja**
- **Vzporedna izvedba agentov**: Istočasno izvajajte več agentov za maksimalno učinkovitost
- **Orkestracija delovnih tokov**: usklajevanje sočasnih operacij ob ohranjanju skladnosti podatkov
- **Optimizacija zmogljivosti**: doseganje znatnega pohitritve s pomočjo vzporednega procesiranja
- **Upravljanje virov**: učinkovita uporaba virov modelov AI med sočasnimi operacijami

### 🏗️ **Napredni vzorci sočasnosti**
- **Obdelava Fork-Join**: razdeli delo med več agentov in združi rezultate
- **Vzporedizem cevovoda**: prekrivajoče se faze izvajanja za stalno prepustnost
- **Uravnoteženje obremenitev**: enakomerno razporejanje dela med razpoložljive vire agentov
- **Sinhronizacijske točke**: usklajevanje sočasnih agentov v kritičnih fazah delovnega toka

### 🏢 **Sočasne poslovne aplikacije**
- **Obdelava velikih količin dokumentov**: istočasna obdelava več dokumentov
- **Analiza vsebine v realnem času**: sočasna analiza prihajajočih podatkovnih tokov
- **Optimizacija paketne obdelave**: maksimiranje prepustnosti za obsežne operacije
- **Večmodalna analiza**: vzporedna obdelava različnih vrst vsebin (besedilo, slike, podatki)

## ⚙️ Zahteve in nastavitev

### 📦 **Zahtevane odvisnosti**

Namestite Agent Framework z zmožnostmi sočasnih delovnih tokov:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundry**

Prijavite se z Azure CLI (`az login`), da lahko `AzureCliCredential` preveri pristnost, nato nastavite podrobnosti projekta Microsoft Foundry.

**Nastavitev okolja (.env datoteka):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Premisleki glede sočasnega procesiranja:**
- **Omejitve hitrosti**: spremljajte omejitve hitrosti Azure OpenAI za sočasne zahteve
- **Uporaba virov**: upoštevajte porabo pomnilnika in CPU z več sočasnimi agenti
- **Ravnanje z napakami**: uvedite robustno obnovo po napakah za vzporedne operacije

### 🏗️ **Arhitektura sočasnega delovnega toka**

```mermaid
graph TD
    A[Začetek delovnega toka] --> B[Hkratno izvajanje]
    B --> C[Skupina agentov 1]
    B --> D[Skupina agentov 2]
    B --> E[Skupina agentov 3]
    C --> F[Združevanje rezultatov]
    D --> F
    E --> F
    F --> G[Končni izhod]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Ključne prednosti:**
- **⚡ Zmogljivost**: znatna pohitritev zaradi vzporedne izvedbe
- **📈 Razširljivost**: obvladovanje povečane obremenitve brez sorazmernega povečanja časa
- **🔄 Učinkovitost**: boljša izraba razpoložljivih računalniških virov
- **🎯 Prepustnost**: obdelajte več dela v enakem času

## 🎨 **Vzorci oblikovanja sočasnih delovnih tokov**

### 🔍 **Cevovod raziskav in analiz**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Delovni tok obdelave podatkov**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Cevovod ustvarjanja vsebin**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Večstopenjsko procesiranje**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Poslovne koristi zmogljivosti**

### ⚡ **Optimizacija prepustnosti**
- **Vzporedna izvedba**: več agentov dela istočasno
- **Izraba virov**: maksimalna učinkovitost razpoložljive zmogljivosti AI modela
- **Zmanjšanje časa**: znatno skrajšanje skupnega časa obdelave
- **Razširljiva arhitektura**: po potrebi enostavno dodajte več sočasnih agentov

### 🛡️ **Zanesljivost in odpornost**
- **Odpornost na napake**: odpovedi posameznih agentov ne ustavijo celotnega delovnega toka
- **Izolacija napak**: težave v eni sočasni veji ne vplivajo na druge
- **Eleganten padec zmogljivosti**: sistem deluje naprej tudi z zmanjšano zmogljivostjo agentov
- **Mehanizmi za obnovitev**: samodejno ponavljanje in ravnanje z napakami za neuspešne operacije

### 📊 **Nadzor in opazovanje**
- **Sledenje sočasni izvedbi**: spremljajte napredek vseh vzporednih operacij
- **Metrične zmogljivosti**: merite pohitritve in izboljšave učinkovitosti
- **Analiza uporabe virov**: optimizirajte razporeditev sočasnih agentov
- **Identifikacija ozkih grl**: poiščite in odpravite omejitve zmogljivosti

Zgradimo visoko zmogljive sočasne AI delovne tokove! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
